# MetroLink Transit Services — Public Transport Insight
**Rounds covered:** Round 2 (EDA & Feature Engineering), Round 3 (Excel analysis support), Round 4 (Business insights).

This project uses a **synthetic dataset** generated for this assessment (seed = 20261031). It is not real operational data. The analysis grain is one trip-stop record, enriched with route, calendar and daily operations attributes.

## Business questions
- Which routes carry the most passengers?
- What are peak travel periods?
- Which stops have high boarding demand?
- How does weekend demand differ?
- Which routes have repeated delays?
- How do events affect ridership?

In [ ]:
# Install if needed: pandas, numpy, matplotlib, seaborn, scikit-learn
from pathlib import Path
import pandas as pd, numpy as np, matplotlib.pyplot as plt, seaborn as sns
from sklearn.preprocessing import LabelEncoder
from sklearn.feature_selection import VarianceThreshold
DATA = Path('data')  # change if running from another folder
trips_raw = pd.read_csv(DATA/'trips.csv', parse_dates=['date'])
board_raw = pd.read_csv(DATA/'boardings_raw.csv')
routes = pd.read_csv(DATA/'routes.csv')
calendar = pd.read_csv(DATA/'calendar.csv', parse_dates=['date'])
operations_raw = pd.read_csv(DATA/'operations_raw.csv', parse_dates=['date'])
print('Trips:', trips_raw.shape, 'Boardings:', board_raw.shape, 'Routes:', routes.shape)
display(trips_raw.head())

## 1. Data quality assessment
The raw boarding file intentionally contains missing stop IDs, duplicate trip-stop rows, inconsistent route labels and negative counts. Operations data contains incomplete delay observations.

In [ ]:
quality = pd.DataFrame({
    'missing_values': board_raw.isna().sum(),
    'duplicate_rows': [board_raw.duplicated().sum()]*len(board_raw),
    'dtype': board_raw.dtypes.astype(str)
})
display(quality)
print('Negative boardings:', (pd.to_numeric(board_raw.boardings, errors='coerce') < 0).sum())
print('Missing delay values:', operations_raw.avg_delay_min.isna().sum())

## 2. Cleaning and integration
Normalize route IDs, remove duplicate trip-stop records, remove negative passenger counts, impute missing stop IDs as `UNKNOWN_STOP`, impute missing delays with route-level medians, then join trips, routes, calendar and operations.

In [ ]:
trips = trips_raw.copy()
board = board_raw.copy()
board['route_id'] = board.route_id.astype(str).str.upper().str.strip()
board['boardings'] = pd.to_numeric(board.boardings, errors='coerce')
board['alightings'] = pd.to_numeric(board.alightings, errors='coerce')
board = board.drop_duplicates(subset=['trip_id','route_id','stop_id'], keep='first')
board = board[(board.boardings >= 0) & (board.alightings >= 0)]
board['stop_id'] = board.stop_id.fillna('UNKNOWN_STOP')
board[['boardings','alightings']] = board[['boardings','alightings']].fillna(0).astype(int)
operations = operations_raw.copy()
operations['route_id'] = operations.route_id.str.upper().str.strip()
operations['avg_delay_min'] = operations.groupby('route_id').avg_delay_min.transform(lambda s:s.fillna(s.median()))
clean = board.merge(trips, on=['trip_id','route_id'], how='left').merge(routes,on='route_id',how='left').merge(calendar,on='date',how='left')
operations['completion_rate'] = operations.completed_trips / operations.scheduled_trips
clean = clean.merge(operations[['route_id','date','scheduled_trips','completed_trips','avg_delay_min','completion_rate']],on=['route_id','date'],how='left')
print('Cleaned shape:', clean.shape, 'Remaining missing:', int(clean.isna().sum().sum()))
display(clean.head())

## 3. Feature engineering
Create time period, day type, passenger totals, event and delay flags, passenger productivity and categorical codes.

In [ ]:
clean['hour'] = pd.to_datetime(clean.start_time, format='%H:%M').dt.hour
clean['period'] = pd.cut(clean.hour,[-1,9,15,23],labels=['AM Peak','Off-Peak','PM Peak']).astype(str)
clean['day_type'] = np.where(clean.is_weekend,'Weekend','Weekday')
clean['total_passengers'] = clean.boardings + clean.alightings
clean['event_flag'] = (clean.event != 'None').astype(int)
clean['delay_flag'] = (clean.avg_delay_min >= 15).astype(int)
clean['passenger_per_km'] = clean.total_passengers / clean.distance_km
for col in ['route_id','corridor','period']:
    clean[col+'_code'] = clean[col].astype('category').cat.codes
clean['is_peak'] = clean.period.isin(['AM Peak','PM Peak']).astype(int)
display(clean[['trip_id','stop_id','period','day_type','total_passengers','event_flag','delay_flag','passenger_per_km']].head())

## 4. Exploratory analysis

In [ ]:
route_demand = clean.groupby(['route_id','route_name'],as_index=False).total_passengers.sum().sort_values('total_passengers',ascending=False)
period_demand = clean.groupby('period',observed=False).total_passengers.sum().sort_values(ascending=False)
stop_demand = clean.groupby('stop_id').boardings.sum().sort_values(ascending=False).head(10)
route_demand.plot.bar(x='route_id',y='total_passengers',legend=False,title='Passenger movements by route',figsize=(9,4)); plt.ylabel('Boardings + alightings'); plt.show()
period_demand.plot.bar(title='Passenger movements by period',figsize=(7,4)); plt.ylabel('Passenger movements'); plt.show()
stop_demand.sort_values().plot.barh(title='Top 10 stops by boardings',figsize=(8,4)); plt.xlabel('Boardings'); plt.show()
display(route_demand, period_demand, stop_demand)

## 5. Weekday/weekend, delay and event analysis

In [ ]:
clean['date'] = pd.to_datetime(clean.date)
daily = clean.groupby(['date','day_type','event']).total_passengers.sum().reset_index()
print(daily.groupby('day_type').total_passengers.mean())
print(daily.assign(is_event=daily.event.ne('None')).groupby('is_event').total_passengers.mean())
delay_by_route = clean.groupby('route_id').avg_delay_min.mean().sort_values(ascending=False)
delay_by_route.plot.bar(title='Average delay by route',figsize=(8,4)); plt.ylabel('Minutes'); plt.axhline(15,ls='--'); plt.show()

## 6. Feature selection and interpretation
The engineered numeric fields are suitable for downstream descriptive or predictive work. A variance filter removes constant fields; this is a basic screening step, not a claim that the remaining fields are causal predictors.

In [ ]:
candidate = clean[['hour','boardings','alightings','distance_km','stop_count','event_flag','delay_flag','passenger_per_km','is_peak','route_id_code','corridor_code','period_code']].fillna(0)
selector = VarianceThreshold(threshold=0.0)
selector.fit(candidate)
selected_features = candidate.columns[selector.get_support()].tolist()
print('Selected non-constant features:', selected_features)
clean.to_csv('data/cleaned_prepared_dataset.csv',index=False)

## 7. Business conclusions
Use the ranked route and stop summaries, peak-period distribution, weekday/weekend comparison, repeated delay counts and event-day uplift to prioritize service planning. Event results are directional because the synthetic sample contains only a small number of event days; validate with real ticketing and AVL data before committing resources.

**Reproducibility:** all raw inputs are included in `data/`; random seed is fixed in the companion generation script. The prepared CSV is also supplied separately.